# Step 01. Dijkstra 알고리즘

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT
>
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

연속 공간을 격자로 이산화한 뒤, **비용이 가장 낮은 노드부터 확장**하는 Dijkstra로 최단 경로를 찾습니다.

## 학습 목표

> **핵심**: '한 번의 확장'을 직접 구현하며, *비용이 가장 낮은 노드부터 확장*해 최단 경로를 찾는 Dijkstra 원리 이해

1. 연속 공간을 격자로 이산화 (섹션 1~5)
   - 해상도와 장애물 팽창(robot radius)으로 grid map 생성
   - 해상도가 만드는 정밀도와 계산량의 트레이드오프 (섹션 14에서 직접 비교)
2. '한 번의 확장' 구현, 전체 탐색 = 그 반복 (핵심, 섹션 6~11)
   - Node는 누적 비용 $g$ 와 parent를 저장, 8방향 이동 비용은 1.0 또는 $\sqrt{2}$ 로 모두 0 이상 (섹션 4~7)
   - 최소 비용 노드 pop -> Closed 확정 -> 이웃을 Open에 추가 (섹션 9, TODO 1~3)
   - parent 역추적으로 최단 경로 복원 (섹션 10, TODO 4)
   - 조각을 모아 전체 탐색 루프 조립 (섹션 11, TODO 5)
3. 실행 결과로 최적성(모든 비용 0 이상)과 한계(목표 무관 사방 탐색 -> 계산량 많음) 확인 (섹션 12~15)

In [ ]:
%matplotlib inline
import sys, os, math, heapq, time
from collections import namedtuple
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../utils/01_global_planning'))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from grid_map import MakeSimpleMap, BuildObstacleMap
from visualization import PlotGridMap, PlotSearchResult
from grid_search import (CalcGridIndex, CalcGridPosition,
                               GetMotionModel, IsValid)
from search_viz import AnimateSearch

## 1. Grid Map 생성

Dijkstra는 **그래프**(점=노드, 점 사이 연결=엣지) 위에서 동작합니다. 격자의 각 칸을 노드로, 이웃 칸으로 가는 이동을 엣지로 삼습니다.
출발(-5,-5)에서 목표(50,50)까지 벽을 피해 경로를 찾습니다.

In [ ]:
obstacle_x, obstacle_y, start_x, start_y, goal_x, goal_y = MakeSimpleMap()
print(f'장애물 {len(obstacle_x)}개,  출발 ({start_x},{start_y})  ->  목표 ({goal_x},{goal_y})')

fig, ax = plt.subplots(figsize=(6, 6))
PlotGridMap(obstacle_x, obstacle_y, start_x, start_y, goal_x, goal_y, ax=ax)
ax.grid(False)
ax.set_title('Simple Grid Map')
plt.tight_layout()
plt.show()

## 2. 격자 파라미터

**Resolution**: 격자 한 칸 크기(작을수록 정밀하지만 느림).
**Robot Radius**: 장애물을 이만큼 팽창시켜 차량을 점으로 단순화.

값을 정하고, 선택한 해상도(2m) 격자를 맵 위에 그려 봅니다. 해상도가 탐색 효율(노드 수, 시간, 경로 길이)에 주는 영향은 **알고리즘을 직접 실행한 뒤** 섹션 14에서 비교합니다.

In [ ]:
RESOLUTION   = 2.0  # 격자 한 칸 크기 [m]
ROBOT_RADIUS = 1.0  # 장애물 팽창 반경 [m]

min_x, max_x = round(min(obstacle_x)), round(max(obstacle_x))
min_y, max_y = round(min(obstacle_y)), round(max(obstacle_y))
x_width = int((max_x - min_x) / RESOLUTION) + 1
y_width = int((max_y - min_y) / RESOLUTION) + 1

print(f'맵 범위  x[{min_x}, {max_x}]  y[{min_y}, {max_y}]')
print(f'해상도 {RESOLUTION} m  ->  격자 {x_width} x {y_width} = {x_width*y_width:,} cells')

def DrawGrid(ax, resolution, color='lightgray', lw=0.5):
    """해상도 resolution의 격자선을 ax에 그린다 (이후 맵 시각화에서 공통 사용)."""
    for xline in np.arange(min_x, max_x + resolution, resolution):
        ax.axvline(xline, color=color, lw=lw, zorder=0)
    for yline in np.arange(min_y, max_y + resolution, resolution):
        ax.axhline(yline, color=color, lw=lw, zorder=0)

fig, ax = plt.subplots(figsize=(6, 6))
DrawGrid(ax, RESOLUTION)
ax.plot(obstacle_x, obstacle_y, '.k', ms=2, alpha=0.5)
ax.plot(start_x, start_y, 'og', ms=10)
ax.plot(goal_x, goal_y, 'xb', ms=10, mew=2)
ax.set_aspect('equal')
ax.grid(False)
ax.set_title(f'{RESOLUTION}m grid on the map  ({x_width} x {y_width} cells)')
plt.tight_layout()
plt.show()

## 3. 장애물 맵 (팽창)

`BuildObstacleMap`이 각 격자 셀을 **통행 가능(free)** 또는 **장애물(blocked)** 로 분류합니다.
이때 장애물을 robot_radius만큼 **팽창**시켜 차량을 점으로 단순화합니다.
robot_radius가 클수록 안전하지만 좁은 통로가 막힐 수 있습니다.

아래 그림은 원본 벽(빨강) 위에 팽창된 통행 불가 셀(회색)을 겹쳐 보여줍니다. `ROBOT_RADIUS`를 바꿔 다시 실행하면 '벽이 두꺼워지는' 팽창 효과를 직접 확인할 수 있습니다.

In [ ]:
# 장애물을 robot_radius만큼 '팽창'시켜 통행 가능/불가 격자 생성
obstacle_map, min_x, min_y, max_x, max_y, x_width, y_width = BuildObstacleMap(obstacle_x, obstacle_y, RESOLUTION, ROBOT_RADIUS)

def BlockedCells(obstacle_map):
    blocked_x, blocked_y = [], []
    for ix in range(len(obstacle_map)):
        for iy in range(len(obstacle_map[0])):
            if obstacle_map[ix][iy]:
                blocked_x.append(CalcGridPosition(ix, min_x, RESOLUTION))
                blocked_y.append(CalcGridPosition(iy, min_y, RESOLUTION))
    return blocked_x, blocked_y

blocked_x, blocked_y = BlockedCells(obstacle_map)

# 출발/목표를 가장 가까운 격자 셀 중심으로 옮김
start_center_x = CalcGridPosition(CalcGridIndex(start_x, min_x, RESOLUTION), min_x, RESOLUTION)
start_center_y = CalcGridPosition(CalcGridIndex(start_y, min_y, RESOLUTION), min_y, RESOLUTION)
goal_center_x = CalcGridPosition(CalcGridIndex(goal_x, min_x, RESOLUTION), min_x, RESOLUTION)
goal_center_y = CalcGridPosition(CalcGridIndex(goal_y, min_y, RESOLUTION), min_y, RESOLUTION)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
DrawGrid(ax, RESOLUTION)
ax.plot(obstacle_x, obstacle_y, '.', color='red', ms=2, alpha=0.5)
ax.plot(blocked_x, blocked_y, 's', color='dimgray', ms=4, alpha=0.85)
ax.plot(start_center_x, start_center_y, 'og', ms=11, zorder=5)
ax.plot(goal_center_x, goal_center_y, 'xb', ms=11, mew=2.5, zorder=5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
ax.set_xlim(min_x - 3, max_x + 3)
ax.set_ylim(min_y - 3, max_y + 3)
ax.set_title(f'Obstacle inflation (robot_radius={ROBOT_RADIUS}m): red=raw wall, gray=blocked')
plt.tight_layout()
plt.show()
print(f'robot_radius={ROBOT_RADIUS}m -> 통행 가능 {x_width*y_width - len(blocked_x)}개 / 장애물(팽창) {len(blocked_x)}개')

## 4. Node - 상태 표현

노드 하나 = 탐색 상태 하나. 저장 정보:

| 필드 | 의미 |
|------|------|
| `x, y` | 격자 인덱스 |
| `cost` | 출발부터 누적 비용 g(n) |
| `parent_index` | 직전 노드 키 (역추적용), 시작은 -1 |

In [ ]:
class Node:
    # 격자 탐색의 기본 단위. (x, y)는 실제 좌표가 아닌 격자 인덱스.
    def __init__(self, x, y, cost, parent_index):
        self.x = x
        self.y = y
        self.cost = cost                   # 출발점부터 누적 비용 g(n)
        self.parent_index = parent_index   # 직전 노드 키 (ix, iy), 시작은 -1
    def __lt__(self, other):
        return self.cost < other.cost      # heapq 비교용

example_node = Node(CalcGridIndex(start_x, min_x, RESOLUTION),
                    CalcGridIndex(start_y, min_y, RESOLUTION), 0.0, parent_index=-1)
print(f'출발 ({start_x},{start_y}) -> 격자 인덱스 ({example_node.x},{example_node.y}), '
      f'cost={example_node.cost}, parent={example_node.parent_index}')

## 5. 좌표 변환 헬퍼

```
CalcGridIndex    : 좌표[m]    -> 격자 인덱스
CalcGridPosition : 격자 인덱스 -> 좌표[m]
```

Open/Closed Set의 **dict 키로는 격자 인덱스 `(ix, iy)` 튜플을 그대로** 사용합니다.
(참고: 실제 구현에서는 속도를 위해 2D 인덱스를 1D 정수로 평탄화하기도 합니다.)

> 아래 출력에서 출발점 (-5, -5)는 (-6, -6)으로 옮겨집니다. -5 는 격자 중심 -6 과 -4 의 정중앙인데, 파이썬 `round`는 정중앙일 때 짝수 쪽으로 반올림하기 때문입니다.

In [ ]:
# 좌표 -> 인덱스 -> 좌표 로 왕복시켜 변환 함수 확인
ix = CalcGridIndex(start_x, min_x, RESOLUTION)
iy = CalcGridIndex(start_y, min_y, RESOLUTION)
back_x = CalcGridPosition(ix, min_x, RESOLUTION)
back_y = CalcGridPosition(iy, min_y, RESOLUTION)
print(f'출발 좌표 ({start_x}, {start_y}) m')
print(f'  -> CalcGridIndex    -> 격자 인덱스 ({ix}, {iy})')
print(f'  -> CalcGridPosition -> 좌표 ({back_x}, {back_y}) m   (해상도 {RESOLUTION}m 격자의 가장 가까운 칸으로 옮겨짐)')

## 6. 이동 모델 (Motion Model)

그래프의 **엣지**. 8방향 이동, 비용은 유클리드 거리 (직선 1.0, 대각선 $\sqrt{2}$).
대각선 비용에 $\sqrt{2}$를 쓰면 대각선 방향의 거리 왜곡이 없어집니다.

> 모든 이동 비용이 **0 이상** 임에 주목하세요. 이것이 Dijkstra가 최적 경로를 보장하기 위한 전제(음수 가중치 없음)입니다.

In [ ]:
motion = GetMotionModel()

def DirLabel(dx, dy):
    # (dx, dy) 부호로 방향 이름 생성
    vertical   = {1: 'U', 0: '', -1: 'D'}[dy]
    horizontal = {1: 'R', 0: '', -1: 'L'}[dx]
    return vertical + horizontal

DIRS = [DirLabel(dx, dy) for dx, dy, _ in motion]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-2.4, 2.4)
ax.set_ylim(-2.4, 2.4)
ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.6)
ax.axvline(0, color='k', lw=0.6)
ax.grid(True, alpha=0.3)
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])

for (dx, dy, cost), label in zip(motion, DIRS):
    color = 'royalblue' if cost == 1.0 else 'darkorange'
    ax.annotate('', xy=(dx * 1.5, dy * 1.5), xytext=(dx * 0.35, dy * 0.35), arrowprops=dict(arrowstyle='->', color=color, lw=2.2))
    ax.text(dx * 1.95, dy * 1.95, f'{label}\n{cost:.2f}', ha='center', va='center', color=color, fontsize=9, fontweight='bold')
    
ax.text(0, 0, 'current', ha='center', va='center', fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='gold', edgecolor='none'))
ax.set_title('8-Direction Motion Model\n(blue = straight 1.0, orange = diagonal sqrt2)', fontsize=11)
plt.tight_layout()
plt.show()

## 7. IsValid - 통행 가능한 이웃인가?

이웃 노드를 만들 때마다 **격자 범위 안 + 장애물 아님**을 확인해야 합니다.
`IsValid`가 이 두 조건을 검사합니다. 아래 `query_x, query_y`(검사할 지점, 초기값=출발)의 8이웃에 적용해 봅니다.
실습 중 `query_x, query_y`를 다른 좌표로 바꿔 보세요. 벽 근처로 옮기면 막힌 이웃이 빨강으로 나타납니다.

In [ ]:
query_x, query_y = start_x, start_y

query = Node(CalcGridIndex(query_x, min_x, RESOLUTION),
             CalcGridIndex(query_y, min_y, RESOLUTION), 0.0, -1)

fig, ax = plt.subplots(figsize=(6, 6))
DrawGrid(ax, RESOLUTION)
ax.plot(obstacle_x, obstacle_y, '.k', ms=2, alpha=0.4)
query_center_x = CalcGridPosition(query.x, min_x, RESOLUTION)
query_center_y = CalcGridPosition(query.y, min_y, RESOLUTION)
ax.plot(query_center_x, query_center_y, 'og', ms=12, zorder=5)

for label, (dx, dy, cost) in zip(DIRS, motion):
    neighbor = Node(query.x + dx, query.y + dy, cost, -1)
    is_passable = IsValid(neighbor, obstacle_map, min_x, min_y, max_x, max_y, RESOLUTION)
    neighbor_x = CalcGridPosition(neighbor.x, min_x, RESOLUTION)
    neighbor_y = CalcGridPosition(neighbor.y, min_y, RESOLUTION)
    ax.plot(neighbor_x, neighbor_y, 'o', color='limegreen' if is_passable else 'red', ms=10, zorder=4)

ax.set_aspect('equal')
ax.grid(False)
ax.set_title(f'Neighbors of ({query_x}, {query_y})  (green=valid, red=blocked)')
plt.tight_layout()
plt.show()

## 8. Open / Closed Set 초기화

- **Open Set**: 탐색 예정 노드(= 탐색 경계). **우선순위 큐(min-heap)로 구현** - 비용이 가장 낮은 노드를 먼저 꺼냅니다. 처음엔 출발 노드 1개.
- **Closed Set**: 비용이 확정된 노드(다시 안 봄). dict로 구현하며, 키는 격자 인덱스 `(ix, iy)`.

**우선순위 큐(priority queue)** 는 넣는 순서와 상관없이, 꺼낼 때 항상 **우선순위가 가장 높은 원소부터** 나오는 대기열입니다. 파이썬 `heapq` 는 리스트를 min-heap 구조로 유지해서 넣기(`heappush`)와 최솟값 꺼내기(`heappop`)를 빠르게(원소 n개일 때 O(log n), 리스트 전체를 뒤져 최솟값을 찾는 O(n)보다 빠름) 처리합니다. 이 실습에서는 `(비용 g, 노드)` 튜플을 넣기 때문에 우선순위가 곧 누적 비용이 되고, `heappop` 한 번이 곧 "아직 확정되지 않은 노드 중 가장 싼 노드 선택"이 됩니다.

In [ ]:
# 출발을 격자 노드로 (연속 좌표 -> 가장 가까운 격자 칸)
start = Node(CalcGridIndex(start_x, min_x, RESOLUTION),
             CalcGridIndex(start_y, min_y, RESOLUTION), 0.0, -1)

priority_queue = [(0.0, start)]   # Open Set (min-heap): 최소 비용부터 꺼냄
closed_set     = {}               # Closed Set: 키 = (ix, iy)

## 9. 한 번의 확장 (핵심)

Dijkstra의 한 스텝:

```
(1) Open Set에서 최소 비용 노드 current 를 꺼낸다                        <- 제공 (heappop)
(2) current 를 Closed Set으로 옮긴다 (비용 확정)                         <- TODO 1
(3) current 의 8이웃 중 (통행 가능 + 아직 Closed 아님)을 Open Set에 추가  <- TODO 2 (이웃 생성) + TODO 3 (push)
```

아래에서 이 한 스텝을 **4번** 반복하며 두 가지를 함께 봅니다.
- **그림**: 각 셀의 숫자가 **누적 비용 g** - 비용이 낮은 셀부터 확정(회색)됩니다.
- **표**: Open Set의 `노드 | g | parent` - 각 노드의 누적 비용 g와 부모(parent)가 한 스텝마다 채워집니다.

> 같은 칸이 더 싼 경로로 Open에 다시 들어올 수 있지만, **가장 싼 것이 먼저 pop**되어 Closed로 확정되므로 이후의 더 비싼 중복은 pop될 때 `closed_set` 검사로 버려집니다.

전체 탐색(섹션 11)은 이 한 스텝을 목표에 도달할 때까지 반복할 뿐입니다.

In [ ]:
def ExpandOnce(priority_queue, closed_set, obstacle_map,
               min_x, min_y, max_x, max_y, resolution):
    # 최소 비용 노드를 꺼내(1) Closed로 옮기고(2) 8이웃을 Open에 추가(3). 꺼낸 노드를 반환(큐가 비면 None).
    while priority_queue:
        _, current = heapq.heappop(priority_queue)   # (1) 최소 비용 노드를 꺼냄
        current_key = (current.x, current.y)

        if current_key in closed_set:        # 이미 확정된 노드의 중복 -> 건너뜀
            continue

        ##### TODO 1 #####
        # 꺼낸 노드를 Closed Set에 확정하기: 비용 확정, 다시 안 봄 (위 markdown의 (2))
        #   - 키: current_key = (ix, iy)
        #   - 값: current 노드
        #################

        closed_set[current_key] = ...

        #################

        for dx, dy, move_cost in GetMotionModel():

            ##### TODO 2 #####
            # 이웃 노드 만들기: current에서 (dx, dy)만큼 이동한 노드 (위 markdown의 (3), 섹션 6 이동 모델)
            #   - 위치     : current.x + dx, current.y + dy
            #   - 누적 비용: current.cost + move_cost (출발부터의 비용 + 이번 한 칸의 이동 비용)
            #   - parent   : current_key (역추적용)
            # 사용함수: Node(x, y, cost, parent_index) / 기능: 탐색 노드 생성 (섹션 4)
            # 확인: 바로 아래 셀(4번 확장) 표의 g 가 1.00 -> 2.00 -> 2.41 ... 처럼 커져야 정상 (1.00 / 1.41 만 반복 -> 누적 누락)
            #################

            neighbor = ...

            #################

            if not IsValid(neighbor, obstacle_map, min_x, min_y, max_x, max_y, resolution):
                continue
            if (neighbor.x, neighbor.y) in closed_set:
                continue

            ##### TODO 3 #####
            # 유효한 이웃을 Open Set에 추가하기: 우선순위 = 누적 비용 (Dijkstra의 핵심)
            #   - 넣을 곳  : priority_queue (= Open Set, 이 함수의 첫 번째 인자)
            #   - 넣을 항목: (neighbor.cost, neighbor) 튜플 -> 비용이 낮은 노드부터 pop됨
            # 사용함수: heapq.heappush(리스트, 항목) / 기능: min-heap 리스트에 항목 추가 (반환값 없음, 리스트 자체가 바뀜)
            # 주의: 노드만 넣으면 -> 위쪽 heappop 줄에서 'cannot unpack non-iterable Node' 에러
            #################

            heapq.heappush(...)

            #################

        return current
    return None                              # 큐가 비면 더 확장할 노드 없음

# --- 확인: 출발 노드 1번 확장 -> Closed 1개(Node), Open 8개((g, Node) 튜플) ---
_pq, _closed = [(0.0, start)], {}
ExpandOnce(_pq, _closed, obstacle_map, min_x, min_y, max_x, max_y, RESOLUTION)
assert isinstance(_closed.get((start.x, start.y)), Node), 'TODO 1 확인: closed_set[current_key] 에 current 노드 저장'
assert len(_pq) == 8 and all(isinstance(item, tuple) and isinstance(item[1], Node) for item in _pq), \
    'TODO 3 확인: (neighbor.cost, neighbor) 튜플을 priority_queue 에 push'
assert sorted(round(item[0], 2) for item in _pq) == [1.0] * 4 + [1.41] * 4, \
    'TODO 2 확인: 누적 비용 = current.cost + move_cost (출발 이웃은 1.00 x4, 1.41 x4)'
assert all(item[1].parent_index == (start.x, start.y) for item in _pq), 'TODO 2 확인: parent = current_key ((ix, iy) 튜플)'
print('✅ ExpandOnce 1회 확인 통과: Closed 1개, Open 8개 (g = 1.00 x4, 1.41 x4)')

In [ ]:
# '한 번의 확장'을 4번 반복하며 탐색 경계가 퍼지는 과정을 관찰
# 재실행해도 처음부터 시작하도록 Open/Closed Set을 다시 초기화 (섹션 8과 동일)
priority_queue = [(0.0, start)]
closed_set = {}

def OpenSnapshot():
    # heap에서 아직 Closed 아닌 노드의 최소비용 항목만 추려 표로
    best = {}
    for _, node in priority_queue:
        key = (node.x, node.y)
        if key in closed_set:
            continue
        if key not in best or node.cost < best[key].cost:
            best[key] = node
    return best

ToX = lambda node: CalcGridPosition(node.x, min_x, RESOLUTION)   # 격자 인덱스 -> 좌표[m] (그림용)
ToY = lambda node: CalcGridPosition(node.y, min_y, RESOLUTION)

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
open_table = {}
for step, ax in enumerate(axes.flat, 1):
    current = ExpandOnce(priority_queue, closed_set, obstacle_map,
                         min_x, min_y, max_x, max_y, RESOLUTION)
    open_table = OpenSnapshot()

    # Open Set 상태표 (노드 | g(누적비용) | parent, 비용 오름차순)를 스텝마다 출력
    print(f'[step {step}] pop ({current.x},{current.y}) g={current.cost:.2f}  ->  Open Set 상태:')
    print(f"  {'node(ix,iy)':<12}{'g':>7}   parent")
    for (ix, iy), node in sorted(open_table.items(), key=lambda it: it[1].cost):
        node_str = f'({ix:>2},{iy:>2})'
        parent = 'start' if node.parent_index == -1 else f'({node.parent_index[0]},{node.parent_index[1]})'
        print(f"  {node_str:<12}{node.cost:>7.2f}   {parent}")
    print()

    for node in closed_set.values():
        ax.plot(ToX(node), ToY(node), 's', color='gray', ms=24, alpha=0.85)
        ax.text(ToX(node), ToY(node), f'{node.cost:.1f}', ha='center', va='center', color='white', fontsize=8)

    for node in open_table.values():
        ax.plot(ToX(node), ToY(node), 's', color='deepskyblue', ms=24, alpha=0.7)
        ax.text(ToX(node), ToY(node), f'{node.cost:.1f}', ha='center', va='center', color='black', fontsize=8)

    ax.plot(ToX(current), ToY(current), '*', color='gold', ms=22, zorder=6, markeredgecolor='k')
    ax.set_aspect('equal')
    ax.set_xlim(-12, 2)
    ax.set_ylim(-12, 2)
    ax.grid(True, alpha=0.2)
    ax.set_title(f'step {step}: pop ({current.x},{current.y}) g={current.cost:.2f}  '
                 f'| closed={len(closed_set)} open={len(open_table)}', fontsize=9)
plt.suptitle('One expansion x4  (number in cell = g, gray=Closed, skyblue=Open, star=popped)', fontsize=12)
plt.tight_layout()
plt.show()

## 10. 경로 역추적 (Path Reconstruction)

탐색이 끝나면 각 노드는 **자신의 직전 노드(parent_index)** 를 기억하고 있습니다.
목표에서 시작해 parent를 따라 거꾸로 올라가면 경로가 복원됩니다 (목표 -> ... -> 시작).

> 탐색 루프(섹션 9)는 비용과 parent만 계산할 뿐, '경로'라는 결과물은 이 역추적이 만들어 냅니다.

In [ ]:
def ReconstructPath(goal, closed_set, min_x, min_y, resolution):
    # goal의 parent_index 체인을 시작(-1)까지 따라가며 좌표를 모은 뒤 뒤집어 반환
    path_x = [CalcGridPosition(goal.x, min_x, resolution)]
    path_y = [CalcGridPosition(goal.y, min_y, resolution)]
    parent = goal.parent_index
    max_steps = len(closed_set) + 1     # 정상 구현이면 체인 길이가 이보다 길 수 없음 (안전장치)
    steps = 0
    while parent != -1:                 # -1 = 시작 노드(부모 없음)
        steps += 1
        if steps > max_steps:
            print('[경고] parent가 갱신되지 않아 역추적을 중단합니다 (TODO 4를 확인하세요)')
            break
        node = closed_set[parent]
        path_x.append(CalcGridPosition(node.x, min_x, resolution))
        path_y.append(CalcGridPosition(node.y, min_y, resolution))

        ##### TODO 4 #####
        # parent를 한 칸 위(부모)로 갱신하기: 경로 체인을 시작(-1)까지 거슬러 올라가기 (위 markdown의 목표 -> ... -> 시작)
        #   - 갱신 대상: parent <- 방금 찾은 node 의 parent_index
        #   - 갱신 값  : node.parent_index (node 의 속성, node.x 처럼 점으로 접근)
        # 주의: parent 는 (ix, iy) 튜플 키, node 는 closed_set[parent] 로 꺼낸 Node 객체 -> parent.parent_index 는 에러
        #################

        parent = ...

        #################

    path_x.reverse()
    path_y.reverse()
    return path_x, path_y

# 작은 예제로 parent 체인 역추적 확인
# (ix,iy) 키로 저장된 closed_set: 시작(2,2) -> (3,2) -> (4,3) 으로 이어진다고 가정
demo_closed = {
    (2, 2): Node(2, 2, 0.00, -1),         # 시작 (부모 없음)
    (3, 2): Node(3, 2, 1.00, (2, 2)),
    (4, 3): Node(4, 3, 2.41, (3, 2)),
}
demo_goal = demo_closed[(4, 3)]
path_x, path_y = ReconstructPath(demo_goal, demo_closed, min_x, min_y, RESOLUTION)
print('parent 체인:  (4,3) -> (3,2) -> (2,2)')
print('역추적 경로 (시작 -> 목표 순, 좌표[m]):')
for x, y in zip(path_x, path_y):
    print(f'   ({x:.0f}, {y:.0f})')
assert [(round(x), round(y)) for x, y in zip(path_x, path_y)] == [(-6, -6), (-4, -6), (-2, -4)], \
    'TODO 4 확인: 기대 경로 (-6,-6) -> (-4,-6) -> (-2,-4)'
print('✅ 역추적 확인 통과 (기대 경로와 일치)')

## 11. 전체 루프 조립

지금까지의 조각(초기화 -> 한 번의 확장 -> 목표 확인 -> 역추적)을 `while` 루프로 묶습니다.
**최적성**: 모든 비용이 **0 이상** 이므로, Closed Set에 들어간 노드의 비용은 더 줄지 않습니다.
- 이유: pop 순간 current 는 Open Set 에서 가장 싼 노드 -> 다른 길로 current 에 오려면 이보다 비싼 노드를 거쳐야 하고, 비용은 더해지기만 함(0 이상) -> 더 싼 길이 있을 수 없음

In [ ]:
DijkstraResult = namedtuple('DijkstraResult', 'path_x path_y closed_x closed_y explore_order')

def Dijkstra(start_x, start_y, goal_x, goal_y, obstacle_x, obstacle_y, resolution, robot_radius):
    obstacle_map, min_x, min_y, max_x, max_y, x_width, y_width = BuildObstacleMap(obstacle_x, obstacle_y, resolution, robot_radius)

    start = Node(CalcGridIndex(start_x, min_x, resolution),
                 CalcGridIndex(start_y, min_y, resolution), 0.0, -1)
    goal  = Node(CalcGridIndex(goal_x, min_x, resolution),
                 CalcGridIndex(goal_y, min_y, resolution), 0.0, -1)

    priority_queue = [(0.0, start)]   # Open Set (MinHeap)
    closed_set = {}                   # Closed Set, 키 = (ix, iy)
    explore_order = []                # 시각화용 탐색 순서

    max_expansions = x_width * y_width * 20   # 정상 구현이면 이보다 훨씬 적게 끝남 (안전장치)
    expansions = 0

    # 전체 탐색 = 섹션 9의 ExpandOnce(한 번의 확장)를 목표 도달까지 반복할 뿐
    while True:   # 목표 도달 또는 큐 소진(current is None)에서 break
        expansions += 1
        if expansions > max_expansions:
            print('[경고] 탐색이 비정상적으로 길어지고 있습니다 (TODO 1의 Closed Set 갱신을 확인하세요) - 강제 종료합니다')
            break

        ##### TODO 5 #####
        # 한 번의 확장 실행하기: 이번에 확정된 노드를 current 에 받기 (섹션 9와 같은 호출)
        #   - 인자  : priority_queue, closed_set, obstacle_map, min_x, min_y, max_x, max_y, resolution
        #   - 반환값: 이번에 pop되어 확정된 노드 (큐가 비면 None)
        # 사용함수: ExpandOnce(...) / 기능: 최소 비용 노드 pop -> Closed 확정 -> 이웃 push
        #################

        current = ...

        #################

        if current is None:                               # 큐 소진 -> 목표 도달 불가
            print('[경고] 목표에 도달하지 못했습니다 (ROBOT_RADIUS 팽창 등으로 통로가 막혔을 수 있습니다)')
            break

        explore_order.append((CalcGridPosition(current.x, min_x, resolution),
                              CalcGridPosition(current.y, min_y, resolution), current.cost))

        if current.x == goal.x and current.y == goal.y:   # 목표 도달 -> 종료
            # 목표 도달 결과를 goal 노드에 기록 (이후 ReconstructPath가 사용)
            goal.parent_index = current.parent_index
            goal.cost = current.cost
            break

    path_x, path_y = ReconstructPath(goal, closed_set, min_x, min_y, resolution)   # 섹션 10의 역추적
    closed_x = [CalcGridPosition(n.x, min_x, resolution) for n in closed_set.values()]
    closed_y = [CalcGridPosition(n.y, min_y, resolution) for n in closed_set.values()]
    return DijkstraResult(path_x, path_y, closed_x, closed_y, explore_order)

## 12. 탐색 실행

탐색 노드 수(전체 셀 대비 %)가 효율 지표입니다. 경로 길이는 A*와 비교해도 동일해야 합니다.

In [ ]:
start_time = time.time()
result = Dijkstra(start_x, start_y, goal_x, goal_y, obstacle_x, obstacle_y, RESOLUTION, ROBOT_RADIUS)
elapsed_ms = (time.time() - start_time) * 1000

path_x, path_y = result.path_x, result.path_y
closed_x, closed_y = result.closed_x, result.closed_y
explore_order = result.explore_order

path_len = sum(math.hypot(path_x[i+1]-path_x[i], path_y[i+1]-path_y[i]) for i in range(len(path_x)-1))
print(f'탐색 노드 {len(closed_x):,}개  ({len(closed_x)/x_width/y_width*100:.0f}% of {x_width*y_width:,})')
print(f'경로 길이 {path_len:.1f} m,  탐색 시간 {elapsed_ms:.0f} ms')
# 기대값 비교 (기본 파라미터 RESOLUTION=2.0, ROBOT_RADIUS=1.0 일 때만)
if RESOLUTION == 2.0 and ROBOT_RADIUS == 1.0:
    ok = len(closed_x) == 1071 and abs(path_len - 127.2) < 0.1
    print('✅ 기대값 일치: 1,071개 / 127.2 m' if ok else
          '❌ 기대값 1,071개 / 127.2 m 와 다름 -> TODO 2 누적 비용(current.cost + move_cost) 확인')

## 13. 탐색 과정 애니메이션

색(plasma)이 낮은 비용(보라)부터 퍼집니다. 목표 방향을 모르므로 **방사형**으로 확산 -> Dijkstra의 비효율 원인.

In [ ]:
# 비용 순으로 퍼지다 목표 도달 후 경로를 역추적
# (임베디드 애니메이션 용량 축소: dpi를 잠시 낮추고 프레임 배치 수(n_batches)를 줄임, 실행 후 dpi 원복)
_prev_dpi = plt.rcParams['figure.dpi']
plt.rcParams['figure.dpi'] = 50
anim_html = AnimateSearch(explore_order, path_x, path_y, obstacle_x, obstacle_y, start_x, start_y, goal_x, goal_y,
                           title='Dijkstra', n_batches=30,
                           grid=(min_x, max_x, min_y, max_y, RESOLUTION)).to_jshtml()
plt.rcParams['figure.dpi'] = _prev_dpi
HTML(anim_html)

## 14. 해상도(Resolution) 비교

이제 알고리즘을 직접 돌려 봤으니, **해상도만 바꿔** 같은 Dijkstra를 실행해 그 영향을 확인합니다.
해상도가 높을수록(격자가 촘촘, 값이 작음) 셀/탐색 노드/시간이 늘지만 경로는 정밀해지고, 낮을수록(격자가 성김, 값이 큼) 빠르지만 거칩니다 - 섹션 2에서 미뤄 둔 **정밀도 <-> 계산량 트레이드오프**입니다.

> [주의] 해상도가 너무 낮으면(격자가 성김, res=4m) 얇은 벽이 격자에서 사라져, 경로가 벽을 그대로 가로지릅니다 (아래 그림의 빨간 선). 표에서 res=4m 경로가 짧은 것은 더 최적이라서가 아니라 이 **이산화 오류(벽 통과)** 때문입니다 - 해상도는 낮을수록 좋은 게 아닙니다.

In [ ]:
resolutions = [1.0, 2.0, 4.0]
results = []
print(f"{'res[m]':>7}{'cells':>10}{'explored':>10}{'path[m]':>10}{'time[ms]':>10}")
print('-' * 47)

for res in resolutions:
    t0 = time.time()
    r = Dijkstra(start_x, start_y, goal_x, goal_y, obstacle_x, obstacle_y, res, ROBOT_RADIUS)
    elapsed = (time.time() - t0) * 1000
    plen = sum(math.hypot(r.path_x[i+1]-r.path_x[i], r.path_y[i+1]-r.path_y[i])
               for i in range(len(r.path_x)-1))
    cells = (int((max_x - min_x) / res) + 1) * (int((max_y - min_y) / res) + 1)
    results.append((res, r, plen, cells))
    print(f'{res:>7.1f}{cells:>10,}{len(r.closed_x):>10,}{plen:>10.1f}{elapsed:>10.0f}')

# 성기면 얇은 벽을 격자가 못 잡아 경로가 벽을 가로지른다 (이산화 오류)
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
for ax, (res, r, plen, cells) in zip(axes, results):
    xw = int((max_x - min_x) / res) + 1
    yw = int((max_y - min_y) / res) + 1

    DrawGrid(ax, res)

    ax.plot(obstacle_x, obstacle_y, '.k', ms=1.5, alpha=0.5)
    ax.plot(r.path_x, r.path_y, '-r', lw=2, zorder=4)
    ax.plot(start_x, start_y, 'og', ms=9)
    ax.plot(goal_x, goal_y, 'xb', ms=9, mew=2)
    used = '   <-- USED' if res == RESOLUTION else ''
    ax.set_aspect('equal')
    ax.grid(False)
    ax.set_title(f'resolution = {res} m{used}\n{xw} x {yw} = {cells:,} cells, path {plen:.0f} m', fontsize=10)
    
plt.suptitle('Resolution vs Grid Granularity  (finer = more cells = precise but slow)', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([f'{res} m' for res, *_ in results], [len(r.closed_x) for _, r, *_ in results],
       color=['#4C72B0', '#55A868', '#C44E52'])
for i, (_, r, *_) in enumerate(results):
    ax.text(i, len(r.closed_x), f'{len(r.closed_x):,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('explored nodes')
ax.set_title('Explored nodes vs resolution (finer = more)')
plt.tight_layout()
plt.show()

## 15. 결과 요약

### Dijkstra 특징

| 항목 | 내용 |
|------|------|
| 최적성 | 보장 (모든 비용 >= 0) |
| 완전성 (길이 있으면 반드시 찾음) | 보장 |
| 방향성 | 없음 -> 전방위 방사형 |
| 한계 | 목표와 무관한 노드도 탐색 |

이 표의 **'최적성 보장'** 과 **'목표 무관 탐색'** 이 바로 **최적성 <-> 계산시간 트레이드오프**입니다.
기본 파라미터(res=2.0m, robot_radius=1.0m) 기준, 섹션 12에서 Dijkstra는 전체 격자의 약 **83%(약 1,071개)** 를 탐색했습니다 - 목표 방향을 모른 채 사방으로 퍼지기 때문입니다.

-> **Step 02 A***: 우선순위를 `g(n)` -> `g(n)+h(n)`로 바꿔 목표 방향에 집중합니다.
같은 맵에서 **탐색 노드가 얼마나 줄어드는지**, 그리고 **경로 길이는 그대로 최적(기본 파라미터 기준 약 127.2 m)** 인지 직접 비교하게 됩니다.

아래 그림은 지금까지의 탐색 결과(하늘색 × = 탐색 영역, 빨강 = 최종 경로)를 다시 한 번 정리해 보여줍니다.

In [ ]:
fig = PlotSearchResult(closed_x, closed_y, path_x, path_y,
                         obstacle_x, obstacle_y, start_x, start_y, goal_x, goal_y, title=f'Dijkstra ({len(closed_x):,} nodes)',
                         grid=(min_x, max_x, min_y, max_y, RESOLUTION))
plt.tight_layout()
plt.show()